In [ ]:
import os
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

In [ ]:
orig_w, orig_h = 200, 200
new_w, new_h = 100, 100
x_min, y_min, x_max, y_max = 40, 60, 120, 160

scale_x = new_w / orig_w
scale_y = new_h / orig_h

new_x_min = int(x_min * scale_x)
new_y_min = int(y_min * scale_y)
new_x_max = int(x_max * scale_x)
new_y_max = int(y_max * scale_y)

print(f"Scaled bounding box:")
print(f"x_min = {new_x_min}, y_min = {new_y_min}")
print(f"x_max = {new_x_max}, y_max = {new_y_max}")

In [ ]:
!kaggle datasets download -d pranavraikokte/covid19-image-dataset --force
import zipfile
with zipfile.ZipFile('covid19-image-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('covid19_dataset')

image_paths = []
for root, dirs, files in os.walk('covid19_dataset'):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

In [ ]:
base_model = MobileNetV2(input_shape=(100, 100, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
outputs = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
selected_paths = image_paths[:3]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, path in enumerate(selected_paths):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (100, 100))

    cv2.rectangle(img_resized, (20, 30), (60, 80), (255, 0, 0), 4)

    axes[i].imshow(img_resized)
    axes[i].set_title(f"Image {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()